In [1]:

! pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [15]:
# Kutubxonalar

import torch
import torch.nn.functional as F
import pandas as pd
import re
from collections import Counter
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import gradio as gr

**Ma’lumotlarni tayyorlash va tahlil qilish**

In [3]:
# Dataset

url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/refs/heads/master/IMDB-Dataset.csv"

df = pd.read_csv(url)
df = df[['review', 'sentiment']]
df.head(10)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
5,"Probably my all-time favorite movie, a story o...",positive
6,I sure would like to see a resurrection of a u...,positive
7,"This show was an amazing, fresh & innovative i...",negative
8,Encouraged by the positive comments about this...,negative
9,If you like original gut wrenching laughter yo...,positive


In [4]:
# Ortiqcha belgilardan tozalash

cleaned_reviews = []

for text in df['review']:
    text = text.replace('<br />', ' ')
    cleaned_reviews.append(text)

df['review'] = cleaned_reviews
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming t...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
# 1000 tadan tasodifiy namuna

pos_df = df[df['sentiment'] == 'positive'].sample(n=1000, random_state=101)
neg_df = df[df['sentiment'] == 'negative'].sample(n=1000, random_state=101)

df_final = pd.concat([pos_df, neg_df]).sample(frac=1, random_state=101).reset_index(drop=True)
df_final['label'] = df_final['sentiment'].map({'positive': 1, 'negative': 0})

**RNN modeli uchun ma’lumotlarni kodlash**

In [6]:
# Lug‘at yaratish

def tokenize(text):
    return re.findall(r'\w+', text.lower())

all_tokens = []
for review in df_final['review']:
    all_tokens.extend(tokenize(review))

counts = Counter(all_tokens)
vocab = {word: i+2 for i, (word, _) in enumerate(counts.most_common(4000))}
vocab['<pad>'] = 0
vocab['<unk>'] = 1

#  Matnni sonli ketma-ketlikka o‘girish

def encode_review(text, vocab, max_len=256):
    tokens = tokenize(text)
    encoded = [vocab.get(token, 1) for token in tokens]
    if len(encoded) < max_len:
        encoded += [0] * (max_len - len(encoded))
    return encoded[:max_len]

df_final['encoded'] = df_final['review'].apply(lambda x: encode_review(x, vocab))

**SentimentRNN modelini qurish va o‘qitish**

In [7]:

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(SentimentRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        return self.sigmoid(self.fc(hidden.squeeze(0)))

X_rnn = torch.LongTensor(list(df_final['encoded']))
y_rnn = torch.FloatTensor(df_final['label'].values).view(-1, 1)
dataset_rnn = DataLoader(TensorDataset(X_rnn, y_rnn), batch_size=32, shuffle=True)

model_rnn = SentimentRNN(len(vocab), 128, 256)
optimizer = torch.optim.Adam(model_rnn.parameters(), lr=0.001)
criterion = nn.BCELoss()

for epoch in range(10):
    total_loss = 0
    for inputs, labels in dataset_rnn:
        optimizer.zero_grad()
        output = model_rnn(inputs)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataset_rnn):.4f}")

Epoch 1, Loss: 0.7019
Epoch 2, Loss: 0.6727
Epoch 3, Loss: 0.6646
Epoch 4, Loss: 0.6980
Epoch 5, Loss: 0.6974
Epoch 6, Loss: 0.6925
Epoch 7, Loss: 0.6912
Epoch 8, Loss: 0.6932
Epoch 9, Loss: 0.6912
Epoch 10, Loss: 0.6950


**Transformer modelini tayyorlash va o‘qitish**

In [8]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# Dataset yaratish
ds = Dataset.from_pandas(df_final[['review', 'label']])
ds = ds.train_test_split(test_size=500, seed=123)

# Tokenizatsiya
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples["review"], padding="max_length", truncation=True, max_length=256)

tokenized_ds = ds.map(preprocess_function, batched=True)

# Modelni yuklash va o‘qitish
model_trans = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    logging_steps=100,
    report_to='none'
)

trainer = Trainer(
    model=model_trans,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"]
)

trainer.train()
print(f"Transformer Eval: {trainer.evaluate()['eval_loss']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.435127
200,0.168565


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Transformer Eval: 0.413855642080307


**Ikkala modelni taqqoslash va xulosa**

In [13]:
# Bashorat funksiyalari

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_rnn.to(device)
model_trans.to(device)

def predict_rnn(text):
    model_rnn.eval()
    encoded = torch.LongTensor(encode_review(text, vocab)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model_rnn(encoded)
        prob = output.item()
    return "positive" if prob > 0.5 else "negative"

def predict_transformer(text):
    model_trans.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    with torch.no_grad():
        logits = model_trans(**inputs).logits
    pred = torch.argmax(logits, dim=1).item()
    return "positive" if pred == 1 else "negative"

# Test qilish
test_reviews = [
    "Bu filmni juda uzoq kutgandim, lekin umuman yoqmadi.",
    "Aktyorlar jamoasi ajoyib, syujet ham juda qiziqarli ekan.",
    "Filmning vizual effektlari yaxshi, ammo hikoyasi zaif.",
    "I had waited for this movie for a long time, but I didn't like it at all.",
    "The cast is wonderful, and the plot is also very interesting.",
    "The visual effects of the movie are good, but the story is weak."
]

for r in test_reviews:
    print(f"Sharh: {r}")
    print(f"RNN: {predict_rnn(r)} | Transformer: {predict_transformer(r)}\n")

Sharh: Bu filmni juda uzoq kutgandim, lekin umuman yoqmadi.
RNN: negative | Transformer: positive

Sharh: Aktyorlar jamoasi ajoyib, syujet ham juda qiziqarli ekan.
RNN: negative | Transformer: positive

Sharh: Filmning vizual effektlari yaxshi, ammo hikoyasi zaif.
RNN: negative | Transformer: positive

Sharh: I had waited for this movie for a long time, but I didn't like it at all.
RNN: negative | Transformer: negative

Sharh: The cast is wonderful, and the plot is also very interesting.
RNN: negative | Transformer: positive

Sharh: The visual effects of the movie are good, but the story is weak.
RNN: negative | Transformer: negative



**Deployment**

In [17]:
interface_rnn = gr.Interface(
    fn=predict_rnn,
    inputs='text',
    outputs='text',
    title='RNN Model',
    description='Sharh kiriting va RNN modelidan foydalanib, uning kayfiyatini bashorat qiling.'
)

interface_rnn.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0e8c5d82950caa381f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:
interface_transformer = gr.Interface(
    fn=predict_transformer,
    inputs='text',
    outputs='text',
    title='Transformer Model',
    description='Sharh kiriting va Transformer modelidan foydalanib, uning kayfiyatini bashorat qiling.'
)
interface_transformer.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a666a7941b46fab8f0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Xulosa**

**O'qitish natijalari va yangi sharhlar ustidagi sinovlar shuni ko'rsatdiki, Transformer (DistilBERT) modeli ushbu vazifa uchun RNN modeliga qaraganda ancha samarali va aqlliroq yechimdir. Transformer modeli o'zining "self-attention" mexanizmi tufayli aniq tushundi, RNN esa kichik hajmdagi ma'lumotlarda o'qitilgani va uzoq bog'liqliklarni saqlay olmagani sababli barcha sharhlarga bir xil (salbiy) baho berdi. Shuning uchun "Cinemania" loyihasi uchun yuqori aniqlik va lingvistik tushunchaga ega bo'lgan Transformer modelini qo'llash tavsiya etiladi.**

In [19]:
# Modelni Saqlash

model_trans.save_pretrained('./transformer_model')
tokenizer.save_pretrained('./transformer_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./transformer_model/tokenizer_config.json',
 './transformer_model/tokenizer.json')